# Laboratorio 7 — Spark ML Lib: salarios en la ENEIC

**CC3066 Data Science — Universidad del Valle de Guatemala — Semestre II 2026**

**Integrantes:** Dilary Cruz · _(agregar integrantes)_

Preguntas que guían el análisis:

1. ¿Qué perfiles de trabajadores asalariados pueden identificarse según su edad, antigüedad y jornada habitual?
2. ¿Qué tan bien puede estimarse el salario mensual de una persona asalariada con características personales y laborales observadas?

**Datos:** bases de *Personas* de la Encuesta Nacional de Empleo e Ingresos Continua (ENEIC) del INE Guatemala.
Trimestres I–IV de 2025 → desarrollo/entrenamiento; trimestre I de 2026 → prueba final.
Fuente: https://www.ine.gob.gt/encuesta-nacional-de-empleo-e-ingresos/

> Este avance cubre la **Parte 1: análisis exploratorio y segmentación (ejercicios 1–4)**.
> Las asociaciones encontradas no son causales ni recomendaciones sobre cuánto debería ganar una persona.
> Todos los resultados son **no ponderados**: describen los registros analizados, no a la población guatemalteca.

In [ ]:
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import openpyxl

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.stat import Correlation
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Módulo compartido del equipo (src/data_pipeline.py): armonización, filtros y diccionarios de códigos
sys.path.append(str(Path("../src").resolve()))
import data_pipeline as dp

spark = (SparkSession.builder
         .appName("Lab7-ENEIC")
         .master("local[*]")
         .config("spark.driver.memory", "3g")
         .config("spark.sql.shuffle.partitions", "16")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

SEED = 42
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Paleta categórica fija (se asigna en orden, nunca cíclica) y colores neutros
PALETA = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
TINTA, TINTA_2, GRILLA = "#1f1f1e", "#6b6a63", "#e4e3de"
MAPA_DIVERGENTE = LinearSegmentedColormap.from_list("azul_rojo", ["#2a78d6", "#f0efec", "#e34948"])
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": TINTA_2, "axes.labelcolor": TINTA, "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "axes.grid": True, "grid.color": GRILLA, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titleweight": "bold", "axes.titlesize": 11, "font.size": 9.5,
})
print("Spark", spark.version)

### Configuración de archivos y códigos

Los archivos se descargaron del sitio del INE y se colocaron en `data/` (no se versionan; ver `.gitignore`).
Los códigos de las variables categóricas se tomaron de los diccionarios de datos (idénticos en los cinco trimestres):

In [ ]:
DATA_DIR = Path("../data")
PARQUET_DIR = DATA_DIR / "parquet"
FORZAR_CONVERSION = False   # True para volver a leer los Excel aunque ya exista el Parquet crudo

ARCHIVOS = dp.ARCHIVOS                          # (periodo_archivo, anio, trimestre_calendario, archivo, uso)
REGISTROS_ESPERADOS = dp.REGISTROS_ESPERADOS    # conteos publicados en el enunciado
COLS_FUENTE = dp.COLS_FUENTE

# Diccionario de datos (P03A03A, P05C16, DOMINIO)
NIVEL_EDUCATIVO, CATEGORIA_OCUPACIONAL = dp.NIVEL_EDUCATIVO, dp.CATEGORIA_OCUPACIONAL
DOMINIO, DESCONOCIDO = dp.DOMINIO, dp.DESCONOCIDO

---
## 1. Carga, armonización y calidad de datos (5 pts)

### 1.1 Estructura de los archivos originales

Antes de leer los datos se inspecciona el encabezado de cada Excel: número de columnas y **posición** de las columnas requeridas.

In [ ]:
def leer_encabezado(ruta):
    wb = openpyxl.load_workbook(ruta, read_only=True)
    ws = wb[wb.sheetnames[0]]
    encabezado = [str(c).strip() if c is not None else "" for c in next(ws.iter_rows(max_row=1, values_only=True))]
    wb.close()
    return encabezado

encabezados = {p: leer_encabezado(DATA_DIR / archivo) for p, _, _, archivo, _ in ARCHIVOS}

estructura = pd.DataFrame({p: {"n_columnas": len(h), **{c: h.index(c) + 1 for c in COLS_FUENTE}}
                           for p, h in encabezados.items()})
print("Número de columnas y posición (1-based) de las columnas requeridas en cada archivo:")
display(estructura)

faltantes_encabezado = {p: [c for c in COLS_FUENTE if c not in h] for p, h in encabezados.items()}
assert not any(faltantes_encabezado.values()), faltantes_encabezado

### 1.2 Conversión Excel → Parquet

Spark no tiene lector nativo de Excel. Cada archivo se lee **individualmente** con pandas/openpyxl, conservando solo las 14 columnas requeridas
y leyéndolas como **texto** (`dtype=str`): un mismo código puede venir como número (`2`) o como texto (`'2'`), así que primero se preserva
el valor tal cual y después se tipifica en Spark de forma explícita. Cada archivo se guarda como Parquet crudo para no volver a leer el Excel.

In [ ]:
def excel_a_parquet(periodo, archivo):
    destino = PARQUET_DIR / "crudo" / periodo
    existia = destino.exists() and not FORZAR_CONVERSION
    if FORZAR_CONVERSION and destino.exists():
        import shutil; shutil.rmtree(destino)
    dp.read_excel_columns(str(DATA_DIR / archivo), str(destino), spark)
    return "ya existía" if existia else "convertido"

for periodo, _, _, archivo, _ in ARCHIVOS:
    print(periodo, archivo, "->", excel_a_parquet(periodo, archivo))

### 1.3 Armonización de tipos e identificación del período

* Texto vacío, `nan`, `None`, etc. se tratan como nulo.
* **Códigos** (`DOMINIO`, `OCUPADOS`, `P03A03A`, `P05C16`, …): se convierten a número y solo se aceptan si son enteros (`'2'`, `2`, `2.0` → `2`).
* **Variables numéricas** (edad, antigüedad, horas, salario, factor): se convierten a `double`.
* `periodo_archivo`, `anio_archivo` y `trimestre_calendario` se asignan **según el archivo de procedencia**, no a partir de `TRIMESTRE`
  (que se conserva sin modificar). También se conserva `archivo_origen`.
* Los cuatro archivos de 2025 se unen con **`unionByName`**.

In [ ]:
def armonizar(periodo, anio, trimestre, archivo):
    return dp.harmonize(spark.read.parquet(str(PARQUET_DIR / "crudo" / periodo)), anio, trimestre, archivo)

armonizados = {p: armonizar(p, a, t, arch) for p, a, t, arch, _ in ARCHIVOS}

base_2025 = None
for p, _, _, _, uso in ARCHIVOS:
    if uso == "entrenamiento":
        base_2025 = armonizados[p] if base_2025 is None else base_2025.unionByName(armonizados[p])
base_2026 = armonizados["2026T1"]

base_2025 = base_2025.cache()
base_2026 = base_2026.cache()

base_2025.printSchema()
base_2025.show(5, truncate=False)

In [ ]:
conteo_original = (base_2025.unionByName(base_2026)
                   .groupBy("periodo_archivo").count()
                   .toPandas().set_index("periodo_archivo").sort_index())
conteo_original["esperado"] = pd.Series(REGISTROS_ESPERADOS)
conteo_original["coincide"] = conteo_original["count"] == conteo_original["esperado"]
conteo_original = conteo_original.rename(columns={"count": "registros_originales"})
display(conteo_original)

print("Valores originales de TRIMESTRE por archivo (se conservan sin modificar):")
display(base_2025.unionByName(base_2026).groupBy("periodo_archivo", "TRIMESTRE").count()
        .orderBy("periodo_archivo", "TRIMESTRE").toPandas())

### 1.4 Faltantes antes de aplicar filtros

Se distinguen dos situaciones: **nulo** (celda vacía en el archivo) y **no convertible** (hay texto pero no es un número/código válido).
Se calculan sobre todos los registros de cada archivo, antes de filtrar.

In [ ]:
MAPA_ANALITICO = {"ANIO": "ANIO", "TRIMESTRE": "TRIMESTRE", "DOMINIO": "dominio", "NUM_HOGAR": "NUM_HOGAR",
                  "NUM_PERSONA": "NUM_PERSONA", "FACTOR": "FACTOR", "OCUPADOS": "ocupado", "P02A03": "edad",
                  "P03A03A": "nivel_educativo", "P05C07A": "antiguedad_anios", "P05C07B": "antiguedad_meses",
                  "P05C16": "categoria_ocupacional", "P05D01": "salario_mensual", "P05H01A": "horas_semanales"}

def faltantes_crudos(periodo):
    filas = dp.missing_report(spark.read.parquet(str(PARQUET_DIR / "crudo" / periodo)))
    return pd.DataFrame([{"periodo_archivo": periodo, "nombre_analitico": MAPA_ANALITICO[f["variable_original"]], **f}
                         for f in filas])

faltantes = pd.concat([faltantes_crudos(p) for p, *_ in ARCHIVOS], ignore_index=True)
faltantes["pct_nulos"] = (100 * faltantes["nulos"] / faltantes["n"]).round(2)

tabla_faltantes_2025 = (faltantes[faltantes["periodo_archivo"].str.startswith("2025")]
                        .groupby(["variable_original", "nombre_analitico"], sort=False)[["n", "nulos", "no_convertibles"]].sum())
tabla_faltantes_2025["pct_nulos"] = (100 * tabla_faltantes_2025["nulos"] / tabla_faltantes_2025["n"]).round(2)
print("Faltantes en la unión de 2025 (antes de filtros):")
display(tabla_faltantes_2025)

print("Porcentaje de nulos por archivo:")
display(faltantes.pivot(index="variable_original", columns="periodo_archivo", values="pct_nulos").loc[COLS_FUENTE])

### 1.5 Unicidad de la clave `periodo_archivo + NUM_HOGAR + NUM_PERSONA`

Si una clave aparece más de una vez **dentro del mismo archivo**, se revisa si las filas son **repeticiones exactas**
(todas las columnas seleccionadas iguales) o **registros en conflicto** (misma clave, contenido distinto). No se usa `dropDuplicates()`.

In [ ]:
CLAVE = ["periodo_archivo", "NUM_HOGAR", "NUM_PERSONA"]
COLS_CONTENIDO = [c for c in base_2025.columns if c not in ("archivo_origen",)]

def revisar_unicidad(df, nombre):
    con_huella = df.withColumn("huella", F.sha2(F.to_json(F.struct(*COLS_CONTENIDO)), 256))
    claves_nulas = df.filter(F.col("NUM_HOGAR").isNull() | F.col("NUM_PERSONA").isNull()).count()
    grupos = (con_huella.groupBy(CLAVE)
              .agg(F.count(F.lit(1)).alias("n_filas"), F.countDistinct("huella").alias("n_versiones"))
              .filter("n_filas > 1").cache())
    resumen = grupos.agg(
        F.count(F.lit(1)).alias("claves_duplicadas"),
        F.coalesce(F.sum("n_filas"), F.lit(0)).alias("filas_involucradas"),
        F.coalesce(F.sum((F.col("n_versiones") == 1).cast("int")), F.lit(0)).alias("claves_repeticion_exacta"),
        F.coalesce(F.sum((F.col("n_versiones") > 1).cast("int")), F.lit(0)).alias("claves_en_conflicto"),
    ).first().asDict()
    print(f"{nombre}: {df.count():,} registros | claves con nulos: {claves_nulas} | "
          f"claves distintas: {df.select(CLAVE).distinct().count():,}")
    print("  ", resumen)
    return grupos

dups_2025 = revisar_unicidad(base_2025, "2025")
dups_2026 = revisar_unicidad(base_2026, "2026T1")

if dups_2025.count() > 0:
    print("Detalle de claves duplicadas en 2025 (primeras 10):")
    ejemplo = dups_2025.orderBy(F.desc("n_versiones")).limit(10)
    base_2025.join(ejemplo, CLAVE).orderBy(*CLAVE).show(30, truncate=False)

In [ ]:
# Contexto longitudinal: la misma pareja NUM_HOGAR + NUM_PERSONA en varios trimestres
# (el identificador del hogar puede no ser estable entre trimestres; esto es solo exploratorio)
apariciones = (base_2025.groupBy("NUM_HOGAR", "NUM_PERSONA")
               .agg(F.countDistinct("periodo_archivo").alias("n_periodos"))
               .groupBy("n_periodos").count().orderBy("n_periodos"))
print("Número de trimestres de 2025 en que aparece cada combinación hogar-persona:")
apariciones.show()

### 1.6 Población analítica y filtros

Se aplican **siempre en el mismo orden** a 2025 y a 2026. Un registro se excluye en el **primer** paso que no cumple
(los valores que no permiten evaluar un criterio —nulos, NaN, infinitos— también se excluyen y se contabilizan en ese paso).
No se imputa el salario y no se recortan salarios extremos.

| Paso | Criterio |
|---|---|
| 1 | Edad finita y ≥ 15 |
| 2 | Ocupado (`OCUPADOS = 1`) |
| 3 | Asalariado (`P05C16` ∈ {1, 2, 3, 4}) |
| 4 | Salario (`P05D01`) numérico, finito y > 0 |
| 5 | Años de antigüedad finitos y ≥ 0 |
| 6 | Meses de antigüedad enteros entre 0 y 11 |
| 7 | Antigüedad = años + meses/12 ≤ edad |
| 8 | Horas habituales > 0 y ≤ 168 |

In [ ]:
PASOS = dp.PASOS_NOMBRES
marcar_exclusion = dp.mark_exclusions

marcado_2025 = marcar_exclusion(base_2025)
marcado_2026 = marcar_exclusion(base_2026)

def tabla_exclusiones(marcado):
    conteos = marcado.groupBy("periodo_archivo", "motivo_exclusion").count().toPandas()
    pivote = conteos.pivot(index="motivo_exclusion", columns="periodo_archivo", values="count").fillna(0).astype(int)
    antes = pivote.sum()
    excluidos = pivote.reindex(PASOS).fillna(0).astype(int)
    filas = [("Registros originales", antes)]
    restantes = antes.copy()
    for paso in excluidos.index:
        filas.append((f"   excluidos en {paso}", -excluidos.loc[paso]))
        restantes = restantes - excluidos.loc[paso]
    filas.append(("Registros finales (población analítica)", restantes))
    tabla = pd.DataFrame({k: v for k, v in filas}).T
    tabla["Total"] = tabla.sum(axis=1)
    return tabla

tabla_filtros = pd.concat([tabla_exclusiones(marcado_2025).rename(columns={"Total": "Total 2025"}),
                           tabla_exclusiones(marcado_2026).drop(columns="Total")], axis=1)
print("Registros por archivo antes y después de cada filtro (valores negativos = excluidos en ese paso):")
display(tabla_filtros)

### 1.7 Variables categóricas, conjunto preparado y guardado en Parquet

Los códigos se validan contra el diccionario. Valores ausentes o no reconocidos se representan como **`DESCONOCIDO`**
(no como cero; el código educativo `0` significa *Ninguno*).

In [ ]:
COLS_FINALES = dp.COLS_FINALES
preparar = dp.prepare_dataset

preparado_2025 = preparar(marcado_2025)
preparado_2026 = preparar(marcado_2026)

preparado_2025.write.mode("overwrite").parquet(str(PARQUET_DIR / "eneic_2025_preparado"))
preparado_2026.write.mode("overwrite").parquet(str(PARQUET_DIR / "eneic_2026T1_preparado"))

# A partir de aquí se trabaja con lo guardado en Parquet
df25 = spark.read.parquet(str(PARQUET_DIR / "eneic_2025_preparado")).cache()
df26 = spark.read.parquet(str(PARQUET_DIR / "eneic_2026T1_preparado")).cache()
print(f"Población analítica 2025: {df25.count():,} registros | 2026T1: {df26.count():,} registros")

for c in ["nivel_educativo", "categoria_ocupacional", "dominio"]:
    n_desc = df25.filter(F.col(c) == DESCONOCIDO).count()
    print(f"{c}: {n_desc} registros DESCONOCIDO en 2025")
df25.printSchema()

### 1.8 Preguntas

**¿Por qué IV de 2025 no puede apilarse por posición de columnas con los otros archivos?**
Porque tiene **302 columnas** en lugar de 270 y un orden distinto (tabla de la sección 1.1): por ejemplo, `P02A03` (edad) está en la posición 9
en IV-2025 y en la 13 en los demás; `P05D01` (salario) en la 98 frente a la 105; `OCUPADOS` en la 298 frente a la 266.
Una unión por posición (`union`) pegaría, por ejemplo, datos de otra pregunta debajo de la edad o del salario sin generar ningún error,
porque Spark solo verifica que el número de columnas y los tipos sean compatibles. `unionByName` alinea las columnas por su nombre,
que es lo que identifica a la variable en el diccionario.

**¿Qué diferencia existe entre un dato ausente porque la pregunta no corresponde y una respuesta no registrada?**
El primero es un **faltante estructural** (*no aplica*): el flujo del cuestionario salta la pregunta. Por ejemplo, a quien no está ocupado no se le
pregunta el salario, la antigüedad ni la categoría ocupacional; por eso esas variables tienen porcentajes altos de nulos en la tabla de 1.4.
No es un problema de calidad y no debe imputarse: la pregunta no tiene respuesta para esa persona.
La **respuesta no registrada** ocurre cuando la pregunta *sí* aplicaba (p. ej. un asalariado ocupado) pero no hay valor: no sabe, no quiso
responder o hubo un error de captura. Ese faltante sí es pérdida de información y puede sesgar resultados si no es aleatorio
(p. ej. si las personas de salarios altos responden menos). Al restringir la población a ocupados asalariados, los faltantes estructurales
de las variables laborales desaparecen, y los que quedan dentro de esa población son en su mayoría no respuesta; esos se excluyen
y se contabilizan en la tabla de filtros.

**¿Por qué una persona observada en dos períodos no debe eliminarse como duplicado del conjunto longitudinal?**
La ENEIC tiene un diseño con rotación: parte de los hogares se entrevista en trimestres consecutivos. Cada fila es una
**observación persona-período**, no una persona. Si la misma persona aparece en T1 y T2, sus respuestas pueden cambiar (salario, horas, empleo)
y cada registro pertenece a la muestra (y al factor de expansión) de su propio trimestre. Eliminarla dejaría a un trimestre sin parte de su
muestra y alteraría su composición. Por eso la unicidad se verifica con la clave **`periodo_archivo + NUM_HOGAR + NUM_PERSONA`**: solo se esperan
claves únicas *dentro* de un mismo archivo. Esto también implica que el número de filas acumulado no es el número de personas distintas.

**¿Por qué el número de registros de la base filtrada no representa a todos los trabajadores del país?**
1. Es una **muestra**: cada registro representa a muchas personas según `FACTOR`, y el análisis es no ponderado.
2. La población se restringió a **asalariados** (gobierno, empresa privada, jornaleros, servicio doméstico) con **salario positivo registrado**:
   quedan fuera trabajadores por cuenta propia, patronos, trabajadores no remunerados y asalariados sin salario reportado.
3. Se excluyeron registros con datos incompletos o inconsistentes (antigüedad, horas).
4. Al unir cuatro trimestres, una misma persona puede contarse varias veces.

**Uso de `FACTOR`.** Es el factor de expansión: el número de personas de la población que representa cada registro según el diseño muestral.
En un análisis poblacional se usaría como peso: totales (`sum(FACTOR)`), medias y proporciones ponderadas y, con la información del diseño
(estratos y conglomerados), errores estándar. Al unir trimestres habría que dividir el factor entre el número de trimestres para obtener
un promedio anual. En este laboratorio se conserva, pero no se usa en el clustering, los modelos ni las métricas.

---
## 2. Estadística descriptiva y exploración (5 pts)

Todas las estadísticas se calculan con Spark sobre **todos** los registros de la población analítica de 2025.
Los percentiles son exactos (`percentile`), no aproximados.

In [ ]:
NUMERICAS = ["salario_mensual", "edad", "antiguedad", "horas_semanales"]

def resumen_numerico(df, cols):
    aggs = []
    for c in cols:
        aggs += [F.count(c).alias(f"{c}|n"), F.mean(c).alias(f"{c}|media"), F.stddev(c).alias(f"{c}|desv_est"),
                 F.min(c).alias(f"{c}|min"), F.max(c).alias(f"{c}|max"),
                 F.expr(f"percentile({c}, array(0.25, 0.5, 0.75, 0.95))").alias(f"{c}|pct"),
                 F.skewness(c).alias(f"{c}|asimetria")]
    fila = df.agg(*aggs).first().asDict()
    filas = []
    for c in cols:
        p25, p50, p75, p95 = fila[f"{c}|pct"]
        filas.append({"variable": c, "n": fila[f"{c}|n"], "media": fila[f"{c}|media"], "mediana": p50,
                      "desv_est": fila[f"{c}|desv_est"], "min": fila[f"{c}|min"], "p25": p25, "p75": p75,
                      "p95": p95, "max": fila[f"{c}|max"], "asimetria": fila[f"{c}|asimetria"]})
    return pd.DataFrame(filas).set_index("variable")

desc_2025 = resumen_numerico(df25, NUMERICAS)
display(desc_2025.round(2))

### 2.1 Distribución de registros entre categorías ocupacionales, niveles educativos y dominios

In [ ]:
def conteo_categoria(df, col):
    t = df.groupBy(col).count().toPandas().sort_values(col).set_index(col)["count"]
    return pd.DataFrame({"n": t, "pct": (100 * t / t.sum()).round(2)})

conteos_cat = {c: conteo_categoria(df25, c) for c in ["categoria_ocupacional", "nivel_educativo", "dominio"]}
for c, t in conteos_cat.items():
    print(c); display(t)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (c, t) in zip(axes, conteos_cat.items()):
    etiquetas = [s.split(" - ", 1)[-1] for s in t.index]
    ax.barh(etiquetas, t["n"], color=PALETA[0], height=0.6)
    for y, (n, p) in enumerate(zip(t["n"], t["pct"])):
        ax.text(n, y, f"  {p:.1f}%", va="center", color=TINTA_2, fontsize=8.5)
    ax.invert_yaxis()
    ax.set_title(c.replace("_", " ").capitalize())
    ax.set_xlabel("Registros (2025, no ponderado)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.set_xlim(0, t["n"].max() * 1.2)
    ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

### 2.2 Distribución del salario: simetría, media y mediana

In [ ]:
s = desc_2025.loc["salario_mensual"]
print(f"Media: Q{s['media']:,.2f} | Mediana: Q{s['mediana']:,.2f} | "
      f"Diferencia: Q{s['media'] - s['mediana']:,.2f} ({100 * (s['media'] / s['mediana'] - 1):.1f}% sobre la mediana)")
print(f"Asimetría (skewness): {s['asimetria']:.2f} | P95: Q{s['p95']:,.2f} | Máximo: Q{s['max']:,.2f}")

# Histogramas calculados en Spark sobre todos los registros (no sobre una muestra)
LIMITE_LINEAL = float(s["p95"]) * 2
cortes_lin = list(np.linspace(0, LIMITE_LINEAL, 61))
_, frec_lin = (df25.select("salario_mensual").filter(F.col("salario_mensual") <= LIMITE_LINEAL)
               .rdd.flatMap(lambda r: r).histogram(cortes_lin))
log_min = math.floor(math.log10(s["min"]) * 10) / 10
log_max = math.ceil(math.log10(s["max"]) * 10) / 10
cortes_log = list(np.arange(log_min, log_max + 0.05, 0.05))
_, frec_log = df25.select(F.log10("salario_mensual")).rdd.flatMap(lambda r: r).histogram(cortes_log)
fuera = df25.filter(F.col("salario_mensual") > LIMITE_LINEAL).count()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
ax = axes[0]
ax.bar(cortes_lin[:-1], frec_lin, width=np.diff(cortes_lin), align="edge", color=PALETA[0], edgecolor="white", linewidth=0.5)
ax.axvline(s["mediana"], color=TINTA, lw=1.5, ls="-", label=f"Mediana Q{s['mediana']:,.0f}")
ax.axvline(s["media"], color=PALETA[1], lw=1.5, ls="--", label=f"Media Q{s['media']:,.0f}")
ax.set_title(f"Salario mensual — escala lineal (hasta 2×P95; {fuera:,} registros por encima no se muestran)")
ax.set_xlabel("Salario mensual (Q)"); ax.set_ylabel("Registros")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend(frameon=False)

ax = axes[1]
ax.bar(cortes_log[:-1], frec_log, width=np.diff(cortes_log), align="edge", color=PALETA[0], edgecolor="white", linewidth=0.5)
ax.axvline(math.log10(s["mediana"]), color=TINTA, lw=1.5, label="Mediana")
ax.axvline(math.log10(s["media"]), color=PALETA[1], lw=1.5, ls="--", label="Media")
ax.set_title("Salario mensual — ESCALA LOGARÍTMICA (log10), todos los registros")
ax.set_xlabel("Salario mensual (Q, eje en escala log10)"); ax.set_ylabel("Registros")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{10 ** x:,.0f}"))
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

### 2.3 Salario mediano por nivel educativo y categoría ocupacional

In [ ]:
def salario_por_grupo(df, col):
    return (df.groupBy(col).agg(
                F.count(F.lit(1)).alias("n"),
                F.expr("percentile(salario_mensual, array(0.25, 0.5, 0.75))").alias("pct"),
                F.mean("salario_mensual").alias("media"))
            .select(col, "n", F.col("pct")[0].alias("p25"), F.col("pct")[1].alias("mediana"),
                    F.col("pct")[2].alias("p75"), "media")
            .orderBy(col).toPandas().set_index(col))

sal_educ = salario_por_grupo(df25, "nivel_educativo")
sal_cat = salario_por_grupo(df25, "categoria_ocupacional")
display(sal_educ.round(1)); display(sal_cat.round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))
for ax, (t, titulo) in zip(axes, [(sal_educ, "Nivel educativo"), (sal_cat, "Categoría ocupacional")]):
    etiquetas = [f"{i.split(' - ', 1)[-1]}\n(n={n:,})" for i, n in zip(t.index, t["n"])]
    x = np.arange(len(t))
    ax.bar(x, t["mediana"], color=PALETA[0], width=0.6, label="Mediana")
    ax.errorbar(x, t["mediana"], yerr=[t["mediana"] - t["p25"], t["p75"] - t["mediana"]],
                fmt="none", ecolor=TINTA_2, capsize=4, lw=1, label="P25–P75")
    ax.set_xticks(x, etiquetas, fontsize=8)
    ax.set_title(f"Salario mensual mediano por {titulo.lower()} (2025)")
    ax.set_ylabel("Salario mensual (Q)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax.grid(axis="x", visible=False)
    ax.legend(frameon=False, loc="upper left")
plt.tight_layout(); plt.show()

### 2.4 Tamaño de la muestra analítica y salario mediano por trimestre

In [ ]:
por_trimestre = (df25.groupBy("periodo_archivo").agg(
                    F.count(F.lit(1)).alias("n"),
                    F.expr("percentile(salario_mensual, 0.5)").alias("mediana"),
                    F.mean("salario_mensual").alias("media"))
                 .orderBy("periodo_archivo").toPandas().set_index("periodo_archivo"))
por_trimestre["registros_originales"] = conteo_original["registros_originales"]
por_trimestre["pct_retenido"] = (100 * por_trimestre["n"] / por_trimestre["registros_originales"]).round(2)
display(por_trimestre.round(1))

# Dos medidas con escalas distintas -> dos gráficos (sin doble eje)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].bar(por_trimestre.index, por_trimestre["n"], color=PALETA[0], width=0.55)
for i, v in enumerate(por_trimestre["n"]):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom", color=TINTA_2, fontsize=8.5)
axes[0].set_title("Registros en la población analítica por trimestre")
axes[0].set_ylabel("Registros"); axes[0].grid(axis="x", visible=False)
axes[1].plot(por_trimestre.index, por_trimestre["mediana"], color=PALETA[0], lw=2, marker="o", ms=8,
             markeredgecolor="white", markeredgewidth=2)
for i, v in enumerate(por_trimestre["mediana"]):
    axes[1].text(i, v, f"Q{v:,.0f}\n", ha="center", va="bottom", color=TINTA_2, fontsize=8.5)
axes[1].set_title("Salario mensual mediano por trimestre")
axes[1].set_ylabel("Salario mensual (Q)")
axes[1].set_ylim(por_trimestre["mediana"].min() * 0.9, por_trimestre["mediana"].max() * 1.08)
plt.tight_layout(); plt.show()

### 2.5 Interpretación

_(ver celda siguiente)_

---
## 3. Relaciones entre variables numéricas (5 pts)

Se ensamblan las cuatro variables con `VectorAssembler` y se calcula la matriz de correlación de Pearson con
`pyspark.ml.stat.Correlation.corr()` sobre todos los registros elegibles de 2025. Como complemento (no sustituye a Pearson)
se calcula Spearman, que es robusta a la fuerte asimetría del salario.

In [ ]:
vector_num = VectorAssembler(inputCols=NUMERICAS, outputCol="vec_num").transform(df25).select("vec_num")
corr_pearson = pd.DataFrame(Correlation.corr(vector_num, "vec_num", "pearson").head()[0].toArray(),
                            index=NUMERICAS, columns=NUMERICAS)
corr_spearman = pd.DataFrame(Correlation.corr(vector_num, "vec_num", "spearman").head()[0].toArray(),
                             index=NUMERICAS, columns=NUMERICAS)
print("Pearson:"); display(corr_pearson.round(3))
print("Spearman (complemento):"); display(corr_spearman.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, (m, titulo) in zip(axes, [(corr_pearson, "Pearson"), (corr_spearman, "Spearman (complemento)")]):
    sns.heatmap(m, annot=True, fmt=".2f", cmap=MAPA_DIVERGENTE, vmin=-1, vmax=1, center=0, square=True,
                linewidths=2, linecolor="white", cbar_kws={"shrink": 0.8}, ax=ax,
                annot_kws={"color": TINTA, "fontsize": 10})
    ax.set_title(f"Correlación {titulo} — 2025 (n={df25.count():,})")
    ax.tick_params(axis="x", rotation=30); ax.grid(False)
plt.tight_layout(); plt.show()

print("Correlación de Pearson con el salario, de mayor a menor |r|:")
print(corr_pearson["salario_mensual"].drop("salario_mensual").sort_values(key=abs, ascending=False).round(3).to_string())

### 3.1 Interpretación

_(ver celda siguiente)_

---
## 4. Segmentación de perfiles con KMeans (10 pts)

**Variables.** La segmentación busca perfiles laborales, así que se usan las tres variables numéricas continuas:
`edad`, `antiguedad` y `horas_semanales`. Se comparan dos variantes:

* **A — sin salario:** edad, antigüedad, horas.
* **B — con salario:** las mismas + `log10(salario)`. Se usa el logaritmo *solo para el clustering* porque, con la escala original,
  la cola larga del salario dominaría la distancia euclidiana incluso después de estandarizar.

Todas las variables se **estandarizan** (`StandardScaler` con media 0 y desviación 1) porque KMeans usa distancias y
las variables tienen escalas muy distintas (años vs. horas vs. quetzales). Se prueban **K = 2, 3, 4 y 5** con semilla fija.

**Criterio para elegir K (acordado como grupo):** mayor coeficiente de **silueta**, siempre que ningún clúster tenga menos del **5 %**
de los registros (para que cada perfil sea interpretable); el codo del WSSSE se usa como apoyo visual.

In [ ]:
VARS_A = ["edad", "antiguedad", "horas_semanales"]
VARS_B = VARS_A + ["log_salario"]
K_VALORES = [2, 3, 4, 5]
MIN_PCT_CLUSTER = 5.0

df25_km = df25.withColumn("log_salario", F.log10("salario_mensual"))
evaluador = ClusteringEvaluator(featuresCol="features", predictionCol="cluster",
                                metricName="silhouette", distanceMeasure="squaredEuclidean")

def preparar_escalado(df, variables):
    etapas = [VectorAssembler(inputCols=variables, outputCol="x_crudo"),
              StandardScaler(inputCol="x_crudo", outputCol="features", withMean=True, withStd=True)]
    modelo = Pipeline(stages=etapas).fit(df)
    return modelo, modelo.transform(df).cache()

def evaluar_k(datos, variante):
    filas, modelos = [], {}
    total = datos.count()
    for k in K_VALORES:
        km = KMeans(k=k, seed=SEED, featuresCol="features", predictionCol="cluster", maxIter=100)
        modelo = km.fit(datos)
        pred = modelo.transform(datos)
        tamanos = modelo.summary.clusterSizes
        filas.append({"variante": variante, "K": k, "silueta": evaluador.evaluate(pred),
                      "WSSSE": modelo.summary.trainingCost,
                      "pct_cluster_min": 100 * min(tamanos) / total,
                      "tamanos": sorted(tamanos, reverse=True)})
        modelos[k] = modelo
    return pd.DataFrame(filas), modelos

escalador_A, datos_A = preparar_escalado(df25_km, VARS_A)
escalador_B, datos_B = preparar_escalado(df25_km, VARS_B)
res_A, modelos_A = evaluar_k(datos_A, "A: sin salario")
res_B, modelos_B = evaluar_k(datos_B, "B: con log-salario")
resultados_km = pd.concat([res_A, res_B], ignore_index=True)
display(resultados_km.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for i, (res, nombre) in enumerate([(res_A, "A: sin salario"), (res_B, "B: con log-salario")]):
    color = PALETA[i]
    axes[0].plot(res["K"], res["silueta"], marker="o", ms=8, lw=2, color=color, label=nombre,
                 markeredgecolor="white", markeredgewidth=2)
    axes[1].plot(res["K"], res["WSSSE"], marker="o", ms=8, lw=2, color=color, label=nombre,
                 markeredgecolor="white", markeredgewidth=2)
axes[0].set_title("Coeficiente de silueta por K (mayor es mejor)"); axes[0].set_xlabel("K"); axes[0].set_ylabel("Silueta")
axes[1].set_title("WSSSE por K (método del codo)"); axes[1].set_xlabel("K"); axes[1].set_ylabel("WSSSE")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
for ax in axes:
    ax.set_xticks(K_VALORES); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
def elegir_k(res):
    validos = res[res["pct_cluster_min"] >= MIN_PCT_CLUSTER]
    return int(validos.sort_values("silueta", ascending=False).iloc[0]["K"])

K_A, K_B = elegir_k(res_A), elegir_k(res_B)
print(f"Variante A (sin salario): K elegido = {K_A}, silueta = {res_A.set_index('K').loc[K_A, 'silueta']:.4f}")
print(f"Variante B (con salario): K elegido = {K_B}, silueta = {res_B.set_index('K').loc[K_B, 'silueta']:.4f}")

# ¿Qué tan parecidas son las particiones A y B? (tabla cruzada)
pred_A = modelos_A[K_A].transform(datos_A).select("NUM_HOGAR", "NUM_PERSONA", "periodo_archivo", F.col("cluster").alias("cluster_A"))
pred_B = modelos_B[K_B].transform(datos_B).select("NUM_HOGAR", "NUM_PERSONA", "periodo_archivo", F.col("cluster").alias("cluster_B"))
print("Tabla cruzada de asignaciones (filas: A sin salario, columnas: B con salario):")
display(pred_A.join(pred_B, ["NUM_HOGAR", "NUM_PERSONA", "periodo_archivo"])
        .groupBy("cluster_A").pivot("cluster_B").count().orderBy("cluster_A").toPandas().set_index("cluster_A"))

### 4.1 ¿Vale la pena incluir el salario?

Se decide **no incluir el salario** en la segmentación final (variante A) y usarlo solo para **describir** los clústeres ex post:

1. **Objetivo del análisis.** La pregunta es qué perfiles de trabajadores existen según edad, antigüedad y jornada; el salario es el
   resultado que después se quiere explicar. Si el salario define los grupos, la conclusión "el clúster X gana más" es circular.
2. **Coherencia con la Parte 2.** El salario es la variable objetivo de los modelos supervisados. Segmentar con él y luego
   comparar salarios entre segmentos mezclaría variable objetivo y descripción (la etiqueta de clúster tampoco puede ser predictor).
3. **Escala.** Aun estandarizado, el salario original es muy asimétrico: unos pocos salarios extremos generan clústeres pequeños
   de valores atípicos. Por eso la variante B necesita el logaritmo.
4. **Evidencia empírica.** Las métricas de la tabla anterior (silueta, tamaño mínimo de clúster y la tabla cruzada) muestran cuánto
   cambia la partición al agregar el salario; la comparación se comenta en la interpretación final.

In [ ]:
K_FINAL = K_A
modelo_final = modelos_A[K_FINAL]
segmentado = modelo_final.transform(datos_A).drop("x_crudo", "features").cache()

total = segmentado.count()
perfil = (segmentado.groupBy("cluster").agg(
            F.count(F.lit(1)).alias("n"),
            F.mean("edad").alias("edad_media"), F.expr("percentile(edad, 0.5)").alias("edad_mediana"),
            F.mean("antiguedad").alias("antig_media"), F.expr("percentile(antiguedad, 0.5)").alias("antig_mediana"),
            F.mean("horas_semanales").alias("horas_media"), F.expr("percentile(horas_semanales, 0.5)").alias("horas_mediana"),
            F.expr("percentile(salario_mensual, 0.5)").alias("salario_mediano"),
            F.mean("salario_mensual").alias("salario_medio"),
            F.mean((F.col("nivel_educativo_cod") >= 5).cast("double")).alias("pct_superior_o_mas"),
            F.mean((F.col("nivel_educativo_cod") <= 2).cast("double")).alias("pct_primaria_o_menos"),
            F.mean((F.col("dominio_cod") == 3).cast("double")).alias("pct_rural"))
          .orderBy("cluster").toPandas().set_index("cluster"))
perfil.insert(1, "pct_registros", 100 * perfil["n"] / total)
for c in ["pct_superior_o_mas", "pct_primaria_o_menos", "pct_rural"]:
    perfil[c] = 100 * perfil[c]

def composicion(col):
    t = segmentado.groupBy("cluster", col).count().toPandas().pivot(index="cluster", columns=col, values="count").fillna(0)
    return (100 * t.div(t.sum(axis=1), axis=0)).round(1)

print(f"Perfil de los {K_FINAL} clústeres (variante A, 2025, n={total:,}):")
display(perfil.round(1))
print("Composición por categoría ocupacional (% dentro de cada clúster):"); display(composicion("categoria_ocupacional"))
print("Composición por nivel educativo (% dentro de cada clúster):"); display(composicion("nivel_educativo"))
print("Composición por dominio (% dentro de cada clúster):"); display(composicion("dominio"))

In [ ]:
# Centroides en unidades estandarizadas (z): cuántas desviaciones estándar se aleja cada clúster de la media global
centros_z = pd.DataFrame(modelo_final.clusterCenters(), columns=VARS_A)
mediana_global = desc_2025.loc["salario_mensual", "mediana"]
centros_z["salario_mediano_rel"] = perfil["salario_mediano"].values / mediana_global - 1

def describir(fila, sal):
    rasgos = []
    rasgos.append("mayores" if fila["edad"] > 0.5 else "jóvenes" if fila["edad"] < -0.5 else "de edad intermedia")
    rasgos.append("con alta antigüedad" if fila["antiguedad"] > 0.5 else
                  "con poca antigüedad" if fila["antiguedad"] < -0.3 else "con antigüedad media")
    rasgos.append("jornada extensa" if fila["horas_semanales"] > 0.5 else
                  "jornada parcial/corta" if fila["horas_semanales"] < -0.5 else "jornada típica")
    rasgos.append(f"salario mediano {'superior' if sal > 0.05 else 'inferior' if sal < -0.05 else 'similar'} "
                  f"al global ({100 * sal:+.0f}%)")
    return "Trabajadores " + ", ".join(rasgos)

for k in range(K_FINAL):
    print(f"Clúster {k} ({perfil.loc[k, 'pct_registros']:.1f}%): {describir(centros_z.loc[k], centros_z.loc[k, 'salario_mediano_rel'])}")

fig, ax = plt.subplots(figsize=(6.5, 0.6 * K_FINAL + 1.4))
sns.heatmap(centros_z[VARS_A].rename(columns={"horas_semanales": "horas"}), annot=True, fmt="+.2f",
            cmap=MAPA_DIVERGENTE, center=0, vmin=-2.5, vmax=2.5, linewidths=2, linecolor="white",
            annot_kws={"color": TINTA}, cbar_kws={"label": "desviaciones estándar"}, ax=ax)
ax.set_title("Centroides estandarizados por clúster"); ax.set_ylabel("Clúster"); ax.grid(False)
plt.tight_layout(); plt.show()

In [ ]:
# Visualización con una muestra de hasta 5,000 registros (las métricas se calcularon con todos)
muestra = (segmentado.sample(fraction=min(1.0, 5500 / total), seed=SEED).limit(5000)
           .select("edad", "antiguedad", "horas_semanales", "salario_mensual", "cluster").toPandas())

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
pares = [("edad", "antiguedad"), ("edad", "horas_semanales"), ("antiguedad", "horas_semanales")]
for ax, (x, y) in zip(axes, pares):
    for k in range(K_FINAL):
        m = muestra[muestra["cluster"] == k]
        ax.scatter(m[x], m[y], s=9, alpha=0.55, color=PALETA[k], label=f"Clúster {k}", linewidths=0)
    ax.set_xlabel(x); ax.set_ylabel(y); ax.set_title(f"{y} vs {x}")
axes[0].legend(frameon=False, markerscale=2.5)
fig.suptitle(f"Clústeres KMeans (K={K_FINAL}) — muestra de {len(muestra):,} registros de 2025", fontweight="bold")
plt.tight_layout(); plt.show()

# Salario por clúster (percentiles calculados sobre todos los registros)
sal_cluster = salario_por_grupo(segmentado, "cluster")
fig, ax = plt.subplots(figsize=(7, 3.6))
x = np.arange(K_FINAL)
ax.bar(x, sal_cluster["mediana"], color=[PALETA[k] for k in range(K_FINAL)], width=0.55)
ax.errorbar(x, sal_cluster["mediana"], yerr=[sal_cluster["mediana"] - sal_cluster["p25"], sal_cluster["p75"] - sal_cluster["mediana"]],
            fmt="none", ecolor=TINTA_2, capsize=4, lw=1)
ax.set_xticks(x, [f"Clúster {k}\n(n={n:,})" for k, n in zip(sal_cluster.index, sal_cluster["n"])])
ax.set_ylabel("Salario mensual (Q)"); ax.set_title("Salario mediano por clúster (barras: P25–P75)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}")); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

### 4.2 Descripción de los clústeres

_(ver celda siguiente)_